# Designing a learning API through the single neuron

**Lecture 12 · Notebook 00 · Core route: 90 minutes · Full laboratory: 150 minutes**

Linear regression, logistic regression, and a linear support vector machine share an affine neuron and a gradient-based training loop. What should be shared in code, and what must remain configurable?

This laboratory separates the estimator interface used by a data scientist from the computational interface later used inside a neural network. A good API makes assumptions visible, prevents invalid combinations, and gives tests stable boundaries.

## Learning objectives

### How to use this notebook

By the end, you should be able to:

- identify the shared affine structure $z=w^\top x+b$;
- explain why an activation and loss jointly determine the output gradient;
- configure affine estimators for regression, probability estimation, and maximum-margin classification;
- distinguish fit/predict from forward/backward;
- specify shape, state, numerical, and randomness contracts; and
- verify an analytical gradient using finite differences.

**Prerequisites:** linear regression, logistic regression, derivatives, NumPy arrays, and train/test separation.

Predict each shape and derivative before running its cell. This notebook is an experimental laboratory; reusable behavior lives in src/rice_dsm/ml/neural.py and executable contracts live in tests/test_neural_teaching_api.py.

## 1. Why this matters: API design belongs in machine learning

An **application programming interface (API)** is the documented boundary through which code constructs a model, fits it, obtains scores or probabilities, detects invalid state, and diagnoses invalid input.

| Stable concept | Configurable policy |
|---|---|
| affine score $z=Xw+b$ | activation $\phi$ |
| observations-by-features shape | loss and target domain |
| train before predict | optimizer and stopping rule |
| fitted parameters | probability threshold |
| validation behavior | random initialization |

A premature “universal model” hides incompatible semantics. Our bounded design supports two canonical pairings first and tests their claims.

A linear SVM adds a third policy: the affine score defines a hyperplane, while hinge loss and weight regularization choose a wide-margin separator. It shares representation with the other models without sharing their statistical meaning.

In [ ]:
from __future__ import annotations

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from rice_dsm.ml.neural import (
    Identity,
    LinearSVM,
    LogisticCrossEntropy,
    Sigmoid,
    SingleNeuron,
    SquaredErrorRegression,
)
from rice_dsm.paths import course_database_path

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 438

## 2. One affine neuron, several meanings

For observation $x_i\in\mathbb R^p$,

$$
z_i=w^\top x_i+b,\qquad a_i=\phi(z_i).
$$

We call $z_i$ the **preactivation** and $a_i$ the **postactivation**.

- Regression uses $\phi(z)=z$; the output may be any real number.
- Binary logistic regression uses $\phi(z)=\sigma(z)=1/(1+e^{-z})$; the output lies in $(0,1)$.

The interval alone does not prove calibration. A probability interpretation also requires a conditional Bernoulli model, appropriate loss, sampling assumptions, and empirical evidence.

In [ ]:
score_grid = np.linspace(-8.0, 8.0, 401)
identity, sigmoid = Identity(), Sigmoid()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(score_grid, identity.forward(score_grid), color="tab:blue")
axes[0].set(title="Identity activation", xlabel="preactivation z", ylabel="postactivation a")
axes[1].plot(score_grid, sigmoid.forward(score_grid), label="sigmoid")
axes[1].plot(score_grid, sigmoid.derivative(score_grid), "--", label="derivative")
axes[1].set(title="Sigmoid output and sensitivity", xlabel="preactivation z")
axes[1].legend()
plt.tight_layout()

## 3. Activation and loss form one mathematical contract

For half squared error with identity activation,

$$
\ell_i=\tfrac12(a_i-y_i)^2
\quad\Longrightarrow\quad
\frac{\partial\ell_i}{\partial z_i}=a_i-y_i.
$$

For binary cross-entropy with sigmoid activation,

$$
\ell_i=-y_i\log a_i-(1-y_i)\log(1-a_i)
\quad\Longrightarrow\quad
\frac{\partial\ell_i}{\partial z_i}=a_i-y_i.
$$

Thus both canonical models use

$$
\nabla_w L=\frac1nX^\top(a-y),\qquad
\frac{\partial L}{\partial b}=\frac1n\sum_i(a_i-y_i).
$$

The final expressions match for different chain-rule reasons. This is genuine shared structure, not permission to combine arbitrary activations and losses.

In [ ]:
regression = SquaredErrorRegression()
classification = LogisticCrossEntropy()

examples = [
    ("identity + squared error", regression, np.array([1.5, -0.5]), np.array([1.0, 0.0])),
    ("sigmoid + cross-entropy", classification, np.array([0.0, np.log(3.0)]), np.array([0.0, 1.0])),
]
for name, objective, scores, targets in examples:
    predictions = objective.prediction(scores)
    gradient = objective.score_gradient(scores, targets)
    print(f"{name:28s} prediction={predictions}, dL/dz={gradient}")
    np.testing.assert_allclose(gradient, predictions - targets)

### A third affine model: the linear support vector machine

Use signed labels $y_i\in\{-1,+1\}$ and score $z_i=w^\top x_i+b$. A soft-margin linear SVM minimizes

$$
J(w,b)=\frac{\lambda}{2}\lVert w\rVert_2^2
+\frac1n\sum_{i=1}^n\max(0,1-y_i z_i).
$$

The hinge term is zero beyond the required margin, linear inside it, and nondifferentiable at $y_i z_i=1$. A valid subgradient with respect to the score is

$$
\frac{\partial \ell_i}{\partial z_i}=
\begin{cases}
-y_i, & y_i z_i<1,\\
0, & y_i z_i>1.
\end{cases}
$$

This is another configuration of the **single-neuron model**, but it is not the same estimator as logistic or linear regression. The models share an affine score; hinge loss also penalizes correctly classified observations inside the unit margin, logistic loss remains smooth and uses every finite-margin observation, and squared loss measures numerical residuals. Architecture is shared, while learning semantics differ.

In [ ]:
signed_margin = np.linspace(-2.0, 3.0, 501)
squared_loss = 0.5 * (1.0 - signed_margin) ** 2
hinge_loss = np.maximum(0.0, 1.0 - signed_margin)
logistic_loss = np.logaddexp(0.0, -signed_margin)

fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(signed_margin, squared_loss, label="squared loss on signed target")
axis.plot(signed_margin, hinge_loss, label="SVM hinge loss", linewidth=2.5)
axis.plot(signed_margin, logistic_loss, label="logistic loss")
axis.axvline(0.0, color="0.5", linestyle=":", label="decision boundary")
axis.axvline(1.0, color="tab:green", linestyle="--", label="unit margin")
axis.set(xlabel="signed margin  y z", ylabel="loss", ylim=(-0.05, 4.8), title="Same affine score, different learning priorities")
axis.legend(ncols=2)
plt.tight_layout()
assert hinge_loss[np.argmin(np.abs(signed_margin - 1.5))] == 0.0
assert logistic_loss[np.argmin(np.abs(signed_margin - 1.5))] > 0.0

### Numerical stability is part of the API

Logistic loss is evaluated from the score as

$$
\ell(z,y)=\log(1+e^z)-yz
=\operatorname{logaddexp}(0,z)-yz.
$$

The logaddexp expression is intentional. It evaluates the softplus term without overflow. A method that works only for moderate scores has an undocumented numerical domain.

In [ ]:
extreme_scores = np.array([-1_000.0, 0.0, 1_000.0])
extreme_targets = np.array([0.0, 1.0, 1.0])
stable_losses = np.logaddexp(0.0, extreme_scores) - extreme_targets * extreme_scores

print("probabilities:", classification.prediction(extreme_scores))
print("finite losses:", stable_losses)
assert np.isfinite(stable_losses).all()

## 4. Two interfaces, two audiences

A standalone **estimator** owns training: fit, decision_function, and predict.

A network **component** participates in a larger computation: forward, backward, and step.

Calling fit independently on hidden neurons is mathematically wrong because hidden neurons have no observed targets. The network loss supplies one gradient that propagates jointly through every component. We reuse activation and derivative behavior—not an independent training loop—as the network building block.

The SVM also exposes why method semantics matter: decision_function returns a signed score, predict returns a class label, and no predict_proba method is promised. A probability requires a separate calibrated model.

## 5. Worked example: one estimator shell on two real tables

We use diabetes progression for regression and breast-cancer diagnosis for classification. The tables come from the repository’s read-only SQLite database. The examples inspect an API; they do not claim that one unregularized neuron is the best scientific model.

In [ ]:
database_uri = f"{course_database_path().as_uri()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    diabetes = pd.read_sql_query(
        "SELECT bmi, bp, s5, disease_progression FROM diabetes_observations",
        connection,
    )
    cancer = pd.read_sql_query(
        """
        SELECT mean_radius, mean_texture, mean_concave_points, diagnosis_code
        FROM breast_cancer_observations
        """,
        connection,
    )

display(diabetes.describe().T)
display(cancer.groupby("diagnosis_code").size().rename("count"))
assert not diabetes.isna().any().any()
assert set(cancer["diagnosis_code"]) == {0, 1}

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    diabetes.drop(columns="disease_progression"),
    diabetes["disease_progression"].to_numpy(),
    test_size=0.25,
    random_state=RANDOM_STATE,
)
reg_scaler = StandardScaler().fit(Xr_train)
Xr_train = reg_scaler.transform(Xr_train)
Xr_test = reg_scaler.transform(Xr_test)
regressor = SingleNeuron(
    SquaredErrorRegression(), learning_rate=0.01, epochs=2_000
).fit(Xr_train, yr_train)
rmse = np.sqrt(np.mean((regressor.predict(Xr_test) - yr_test) ** 2))

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.drop(columns="diagnosis_code"),
    cancer["diagnosis_code"].to_numpy(dtype=float),
    test_size=0.25,
    stratify=cancer["diagnosis_code"],
    random_state=RANDOM_STATE,
)
cls_scaler = StandardScaler().fit(Xc_train)
Xc_train = cls_scaler.transform(Xc_train)
Xc_test = cls_scaler.transform(Xc_test)
classifier = SingleNeuron(
    LogisticCrossEntropy(), learning_rate=0.05, epochs=2_000
).fit(Xc_train, yc_train)
probabilities = classifier.predict(Xc_test)
accuracy = np.mean((probabilities >= 0.5) == yc_test)

print(f"Regression test RMSE: {rmse:.2f}")
print(f"Classification test accuracy: {accuracy:.3f}")

svm_targets = np.where(yc_train == 1.0, 1.0, -1.0)
svm_test_targets = np.where(yc_test == 1.0, 1.0, -1.0)
margin_classifier = LinearSVM(
    regularization_strength=0.02, learning_rate=0.02, epochs=3_000
).fit(Xc_train, svm_targets)
svm_accuracy = np.mean(margin_classifier.predict(Xc_test) == svm_test_targets)
print(f"Linear SVM test accuracy: {svm_accuracy:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(regressor.loss_history_)
axes[0].set(title="Regression optimization", xlabel="update", ylabel="half MSE", yscale="log")
axes[1].plot(classifier.loss_history_, color="tab:orange")
axes[1].set(title="Classification optimization", xlabel="update", ylabel="cross-entropy", yscale="log")
axes[2].plot(margin_classifier.loss_history_, color="tab:green")
axes[2].set(title="Maximum-margin optimization", xlabel="update", ylabel="regularized hinge objective", yscale="log")
plt.tight_layout()
assert regressor.loss_history_[-1] < regressor.loss_history_[0]
assert classifier.loss_history_[-1] < classifier.loss_history_[0]
assert margin_classifier.loss_history_[-1] < margin_classifier.loss_history_[0]

A falling training loss is optimization evidence, not generalization evidence. Held-out metrics answer a different question. Accuracy at threshold $0.5$ is only one decision summary; unequal harms, imbalance, ranking, and calibration require additional evidence.

Changing the objective did not change validation, the affine score, optimizer loop, fitted-state handling, or prediction call. It did change target meaning, output range, loss, and interpretation.

The linear SVM produces the same kind of affine score but optimizes a regularized hinge objective. Its score is not a probability, and the points on or inside the margin—not every observation equally—determine its local subgradient.

## 6. Debugging with gradient checks and common failure modes

For scalar loss $J(\theta)$, a central difference gives

$$
\frac{\partial J}{\partial\theta_j}
\approx\frac{J(\theta+he_j)-J(\theta-he_j)}{2h}.
$$

A gradient check is a development test on a tiny deterministic problem—not a training method and not proof that the scientific objective is appropriate.

In [ ]:
features = np.array([[0.2, -0.4], [1.1, 0.3], [-0.7, 0.8]])
targets = np.array([1.0, 0.0, 1.0])
weights = np.array([0.3, -0.2])
bias = 0.1

def mean_loss(candidate: np.ndarray) -> float:
    return classification.mean_loss(features @ candidate + bias, targets)

scores = features @ weights + bias
analytical = features.T @ classification.score_gradient(scores, targets) / len(features)
step = 1e-6
numerical = np.empty_like(weights)
for index in range(len(weights)):
    direction = np.zeros_like(weights)
    direction[index] = step
    numerical[index] = (mean_loss(weights + direction) - mean_loss(weights - direction)) / (2 * step)

print("analytical:", analytical)
print("numerical: ", numerical)
np.testing.assert_allclose(analytical, numerical, rtol=1e-5)

## 7. Professional contracts and failure behavior

A useful model API documents and tests shape, dtype, finiteness, target semantics, fitted state, randomness, numerical extremes, and return behavior. Type hints state machine-readable structure; NumPy-style docstrings explain shapes and semantics; runtime validation enforces important boundaries.

Trailing underscores identify learned state such as weights_. Probability threshold policy remains separate from probability estimation.

In [ ]:
try:
    SingleNeuron(SquaredErrorRegression()).predict(np.ones((2, 1)))
except RuntimeError as error:
    print(type(error).__name__ + ":", error)

try:
    SingleNeuron(LogisticCrossEntropy()).fit(np.ones((3, 1)), [0, 1, 2])
except ValueError as error:
    print(type(error).__name__ + ":", error)

## Professional practice and practice ladder

| Data scientist asks | Software engineer asks |
|---|---|
| What quantity does the output estimate? | Which objective encodes that meaning? |
| Was scaling fit only on training data? | Which object owns preprocessing state? |
| Does optimization converge? | Is the gradient independently checked? |
| Which threshold matches decision cost? | Is threshold policy separate from estimation? |

**Guided:** add a finite-difference check for the bias; require relative error below $10^{-5}$.

**Independent:** add a label adapter with a validated threshold; test that two thresholds can produce different decisions.

**Extension:** design regularization, explicitly deciding whether the intercept is penalized.

**Contribution:** add one justified activation or objective to the package with derivation, NumPy-style documentation, boundary tests, and gradient evidence.

## Retrieval practice, takeaway, and further reading

1. What do $z$ and $a$ denote?
2. Why do two canonical models produce $d\ell/dz=a-y$?
3. Why should a hidden neuron not expose independent fit behavior?
4. What does a gradient check establish, and what does it not?
5. Which behavior belongs in tests rather than notebook output?

**Takeaway:** share genuine mathematical structure, keep statistical policy explicit, and design interfaces around valid states.

Notebook 01 assembles literal neurons into layers, derives backpropagation along individual paths, and only then replaces those neurons with matrix operations.

**Further reading:** [Python data model](https://docs.python.org/3/reference/datamodel.html), NumPy documentation, and the cited primary neural-network literature.